# SGPO Training Experiment - GPU Notebook

This notebook runs the Sheaf-Geodesic Policy Optimization (SGPO) experiments on a GPU-enabled cloud instance.

**Deployment Options:**
1. **Modal.com** - Python-native, pay-per-second (recommended)
2. **Google Colab** - Free tier available, easy to use
3. **AWS SageMaker** - Full control, enterprise features

## Setup

In [ ]:
# Check if running on Colab\nimport sys\nIN_COLAB = 'google.colab' in sys.modules\n\nif IN_COLAB:\n    # Clone the repository\n    !git clone https://github.com/MikeHLee/ai_research.git\n    %cd ai_research/topics/high_dimensional_reward_spaces\n    %pip install -q torch matplotlib numpy\nelse:\n    # Local or Modal execution\n    import os\n    os.chdir(os.path.dirname(os.path.abspath('__file__')))\n    %cd ..

In [ ]:
# Core imports
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Normal
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional
import json
import time

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Environment Definition

2D Navigation with Hazard Regions (Black Holes)

In [ ]:
class HazardNavigationEnv:
    """2D navigation with hazard regions modeled as black holes."""
    
    def __init__(
        self,
        goal_pos: np.ndarray = np.array([2.0, 2.0]),
        hazards: List[Tuple[np.ndarray, float]] = None,
        max_steps: int = 200,
        dt: float = 0.1,
    ):
        self.goal_pos = goal_pos
        self.hazards = hazards or [
            (np.array([1.0, 0.5]), 0.3),
            (np.array([0.5, 1.5]), 0.25),
            (np.array([1.5, 1.0]), 0.35),
        ]
        self.max_steps = max_steps
        self.dt = dt
        self.action_bound = 1.0
        self.reset()
    
    def reset(self) -> Tuple[np.ndarray, Dict]:
        self.pos = np.array([0.0, 0.0])
        self.vel = np.array([0.0, 0.0])
        self.step_count = 0
        return self._get_obs(), {}
    
    def _get_obs(self) -> np.ndarray:
        return np.concatenate([self.pos, self.vel]).astype(np.float32)
    
    def step(self, action: np.ndarray) -> Tuple[np.ndarray, float, bool, bool, Dict]:
        action = np.clip(action, -self.action_bound, self.action_bound)
        
        self.vel = self.vel + action * self.dt
        self.vel = np.clip(self.vel, -2.0, 2.0)
        self.pos = self.pos + self.vel * self.dt
        self.step_count += 1
        
        in_hazard = False
        cost = 0.0
        for center, radius in self.hazards:
            if np.linalg.norm(self.pos - center) < radius:
                in_hazard = True
                cost = 1.0
                break
        
        dist_to_goal = np.linalg.norm(self.pos - self.goal_pos)
        reward = -0.1 * dist_to_goal
        if dist_to_goal < 0.2:
            reward += 10.0
        if in_hazard:
            reward -= 5.0
        
        terminated = dist_to_goal < 0.2
        truncated = self.step_count >= self.max_steps
        
        return self._get_obs(), reward, terminated, truncated, {'cost': cost, 'in_hazard': in_hazard}

## Riemannian Metric (Black Hole Model)

In [ ]:
class RiemannianMetric(nn.Module):
    """Learnable Riemannian metric with Schwarzschild-like singularities at hazards."""
    
    def __init__(
        self,
        hazard_centers: List[np.ndarray],
        hazard_radii: List[float],
        base_metric: float = 1.0,
        severity: float = 10.0,
        sharpness: float = 2.0,
    ):
        super().__init__()
        self.register_buffer('hazard_centers', torch.tensor(np.array(hazard_centers), dtype=torch.float32))
        self.register_buffer('hazard_radii', torch.tensor(hazard_radii, dtype=torch.float32))
        
        self.base_metric = nn.Parameter(torch.tensor(base_metric))
        self.severity = nn.Parameter(torch.tensor(severity))
        self.sharpness = nn.Parameter(torch.tensor(sharpness))
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.dim() == 1:
            x = x.unsqueeze(0)
        pos = x[:, :2]
        batch_size = pos.shape[0]
        
        g = torch.ones(batch_size, 1, device=x.device) * F.softplus(self.base_metric)
        
        for i in range(len(self.hazard_radii)):
            center = self.hazard_centers[i]
            radius = self.hazard_radii[i]
            dist = torch.norm(pos - center.unsqueeze(0), dim=1, keepdim=True)
            event_horizon = radius * 0.8
            margin = torch.clamp(dist - event_horizon, min=0.01)
            schwarzschild = F.softplus(self.severity) / (margin ** F.softplus(self.sharpness))
            g = g + schwarzschild
        
        return g

## Neural Network Policies

In [ ]:
class Actor(nn.Module):
    def __init__(self, obs_dim: int, act_dim: int, hidden_dim: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, act_dim),
        )
        self.log_std = nn.Parameter(torch.zeros(act_dim) - 0.5)
    
    def forward(self, obs: torch.Tensor) -> Normal:
        if obs.dim() == 1:
            obs = obs.unsqueeze(0)
        mu = self.net(obs)
        std = torch.exp(self.log_std.clamp(-20, 2))
        return Normal(mu, std)
    
    def get_action(self, obs: torch.Tensor):
        dist = self.forward(obs)
        action = dist.sample()
        log_prob = dist.log_prob(action).sum(-1)
        return action, log_prob


class Critic(nn.Module):
    def __init__(self, obs_dim: int, hidden_dim: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1),
        )
    
    def forward(self, obs: torch.Tensor) -> torch.Tensor:
        if obs.dim() == 1:
            obs = obs.unsqueeze(0)
        return self.net(obs)

## Training Algorithms: PPO, CPO, SGPO

In [ ]:
@dataclass
class TrainingConfig:
    gamma: float = 0.99
    lam: float = 0.97
    clip_ratio: float = 0.2
    actor_lr: float = 3e-4
    critic_lr: float = 1e-3
    metric_lr: float = 1e-2
    train_iters: int = 10
    cost_limit: float = 0.1


def compute_gae(rewards, values, dones, gamma=0.99, lam=0.97):
    advantages, returns = [], []
    gae = 0
    for t in reversed(range(len(rewards))):
        next_value = 0 if t == len(rewards) - 1 else values[t + 1]
        delta = rewards[t] + gamma * next_value * (1 - dones[t]) - values[t]
        gae = delta + gamma * lam * (1 - dones[t]) * gae
        advantages.insert(0, gae)
        returns.insert(0, gae + values[t])
    return torch.tensor(advantages, dtype=torch.float32), torch.tensor(returns, dtype=torch.float32)


def train_episode_ppo(env, actor, critic, actor_optim, critic_optim, config, device):
    obs, _ = env.reset()
    observations, actions, log_probs, rewards, values, dones, costs = [], [], [], [], [], [], []
    
    done = False
    while not done:
        obs_t = torch.tensor(obs, dtype=torch.float32, device=device)
        with torch.no_grad():
            action, log_prob = actor.get_action(obs_t)
            value = critic(obs_t)
        
        next_obs, reward, terminated, truncated, info = env.step(action.cpu().numpy().flatten())
        done = terminated or truncated
        
        observations.append(obs_t)
        actions.append(action)
        log_probs.append(log_prob)
        rewards.append(reward)
        values.append(value.item())
        dones.append(done)
        costs.append(info.get('cost', 0))
        obs = next_obs
    
    advantages, returns = compute_gae(rewards, values, dones, config.gamma, config.lam)
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
    advantages = advantages.to(device)
    returns = returns.to(device)
    
    observations = torch.stack(observations)
    actions = torch.stack(actions).squeeze(1)
    old_log_probs = torch.stack(log_probs)
    
    for _ in range(config.train_iters):
        dist = actor(observations)
        new_log_probs = dist.log_prob(actions).sum(-1)
        ratio = torch.exp(new_log_probs - old_log_probs)
        
        surr1 = ratio * advantages
        surr2 = torch.clamp(ratio, 1 - config.clip_ratio, 1 + config.clip_ratio) * advantages
        actor_loss = -torch.min(surr1, surr2).mean()
        
        actor_optim.zero_grad()
        actor_loss.backward()
        actor_optim.step()
        
        critic_loss = F.mse_loss(critic(observations).squeeze(), returns)
        critic_optim.zero_grad()
        critic_loss.backward()
        critic_optim.step()
    
    return {'return': sum(rewards), 'cost': sum(costs), 'violations': sum(1 for c in costs if c > 0)}


def train_episode_gpo(env, actor, critic, metric, actor_optim, critic_optim, metric_optim, config, device):
    obs, _ = env.reset()
    observations, actions, log_probs, rewards, values, dones, costs = [], [], [], [], [], [], []
    
    done = False
    while not done:
        obs_t = torch.tensor(obs, dtype=torch.float32, device=device)
        with torch.no_grad():
            action, log_prob = actor.get_action(obs_t)
            value = critic(obs_t)
        
        next_obs, reward, terminated, truncated, info = env.step(action.cpu().numpy().flatten())
        done = terminated or truncated
        
        observations.append(obs_t)
        actions.append(action)
        log_probs.append(log_prob)
        rewards.append(reward)
        values.append(value.item())
        dones.append(done)
        costs.append(info.get('cost', 0))
        obs = next_obs
    
    advantages, returns = compute_gae(rewards, values, dones, config.gamma, config.lam)
    observations = torch.stack(observations)
    
    # Riemannian advantage: A_geo = A / sqrt(g(x))
    with torch.no_grad():
        g = metric(observations).squeeze()
    riemannian_advantages = advantages.to(device) / torch.sqrt(g + 1e-8)
    riemannian_advantages = (riemannian_advantages - riemannian_advantages.mean()) / (riemannian_advantages.std() + 1e-8)
    returns = returns.to(device)
    
    actions = torch.stack(actions).squeeze(1)
    old_log_probs = torch.stack(log_probs)
    
    for _ in range(config.train_iters):
        dist = actor(observations)
        new_log_probs = dist.log_prob(actions).sum(-1)
        ratio = torch.exp(new_log_probs - old_log_probs)
        
        surr1 = ratio * riemannian_advantages
        surr2 = torch.clamp(ratio, 1 - config.clip_ratio, 1 + config.clip_ratio) * riemannian_advantages
        actor_loss = -torch.min(surr1, surr2).mean()
        
        actor_optim.zero_grad()
        actor_loss.backward()
        actor_optim.step()
        
        critic_loss = F.mse_loss(critic(observations).squeeze(), returns)
        critic_optim.zero_grad()
        critic_loss.backward()
        critic_optim.step()
    
    # Train metric
    cost_tensor = torch.tensor(costs, dtype=torch.float32, device=device)
    pred_risk = metric(observations).squeeze()
    metric_target = 1.0 + 10.0 * cost_tensor
    metric_loss = F.mse_loss(pred_risk, metric_target)
    
    metric_optim.zero_grad()
    metric_loss.backward()
    metric_optim.step()
    
    return {'return': sum(rewards), 'cost': sum(costs), 'violations': sum(1 for c in costs if c > 0)}

## Run Experiment

In [ ]:
def run_experiment(n_episodes=300, seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    
    hazards = [
        (np.array([1.0, 0.5]), 0.3),
        (np.array([0.5, 1.5]), 0.25),
        (np.array([1.5, 1.0]), 0.35),
    ]
    config = TrainingConfig()
    results = {}
    
    # PPO
    print("Training PPO...")
    torch.manual_seed(seed)
    env_ppo = HazardNavigationEnv(hazards=hazards)
    actor_ppo = Actor(4, 2).to(device)
    critic_ppo = Critic(4).to(device)
    actor_optim = torch.optim.Adam(actor_ppo.parameters(), lr=config.actor_lr)
    critic_optim = torch.optim.Adam(critic_ppo.parameters(), lr=config.critic_lr)
    
    ppo_returns, ppo_violations = [], []
    for ep in range(n_episodes):
        m = train_episode_ppo(env_ppo, actor_ppo, critic_ppo, actor_optim, critic_optim, config, device)
        ppo_returns.append(m['return'])
        ppo_violations.append(m['violations'])
        if (ep + 1) % 50 == 0:
            print(f"  Episode {ep+1}: Return={np.mean(ppo_returns[-50:]):.1f}, Violations={np.mean(ppo_violations[-50:]):.1f}")
    results['PPO'] = {'returns': ppo_returns, 'violations': ppo_violations}
    
    # SGPO
    print("\nTraining SGPO...")
    torch.manual_seed(seed)
    env_gpo = HazardNavigationEnv(hazards=hazards)
    actor_gpo = Actor(4, 2).to(device)
    critic_gpo = Critic(4).to(device)
    metric = RiemannianMetric(
        hazard_centers=[h[0] for h in hazards],
        hazard_radii=[h[1] for h in hazards],
    ).to(device)
    
    actor_optim = torch.optim.Adam(actor_gpo.parameters(), lr=config.actor_lr)
    critic_optim = torch.optim.Adam(critic_gpo.parameters(), lr=config.critic_lr)
    metric_optim = torch.optim.Adam(metric.parameters(), lr=config.metric_lr)
    
    gpo_returns, gpo_violations = [], []
    for ep in range(n_episodes):
        m = train_episode_gpo(env_gpo, actor_gpo, critic_gpo, metric, actor_optim, critic_optim, metric_optim, config, device)
        gpo_returns.append(m['return'])
        gpo_violations.append(m['violations'])
        if (ep + 1) % 50 == 0:
            print(f"  Episode {ep+1}: Return={np.mean(gpo_returns[-50:]):.1f}, Violations={np.mean(gpo_violations[-50:]):.1f}")
    results['SGPO'] = {'returns': gpo_returns, 'violations': gpo_violations}
    
    return results, metric, hazards

# Run
start_time = time.time()
results, metric, hazards = run_experiment(n_episodes=300)
print(f"\nTotal time: {time.time() - start_time:.1f}s")

## Visualize Results

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
window = 20
colors = {'PPO': 'blue', 'SGPO': 'green'}

# Returns
ax1 = axes[0, 0]
for method in ['PPO', 'SGPO']:
    returns = results[method]['returns']
    smoothed = np.convolve(returns, np.ones(window)/window, mode='valid')
    ax1.plot(smoothed, label=method, color=colors[method], linewidth=2)
ax1.set_title('Episode Returns', fontsize=12)
ax1.set_xlabel('Episode')
ax1.set_ylabel('Return')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Violations
ax2 = axes[0, 1]
for method in ['PPO', 'SGPO']:
    violations = results[method]['violations']
    smoothed = np.convolve(violations, np.ones(window)/window, mode='valid')
    ax2.plot(smoothed, label=method, color=colors[method], linewidth=2)
ax2.set_title('Hazard Violations', fontsize=12)
ax2.set_xlabel('Episode')
ax2.set_ylabel('Violations')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Metric field
ax3 = axes[1, 0]
x = np.linspace(-0.5, 3, 50)
y = np.linspace(-0.5, 3, 50)
X, Y = np.meshgrid(x, y)
Z = np.zeros_like(X)
for i in range(len(x)):
    for j in range(len(y)):
        pos = torch.tensor([X[j, i], Y[j, i], 0, 0], dtype=torch.float32, device=device)
        with torch.no_grad():
            Z[j, i] = np.log(metric(pos).item() + 1)

contour = ax3.contourf(X, Y, Z, levels=20, cmap='hot')
plt.colorbar(contour, ax=ax3, label='log(g(x) + 1)')
for center, radius in hazards:
    circle = plt.Circle(center, radius, fill=False, color='white', linewidth=2)
    ax3.add_patch(circle)
ax3.plot(2.0, 2.0, 'g*', markersize=15, label='Goal')
ax3.plot(0.0, 0.0, 'wo', markersize=10, label='Start')
ax3.set_title('Learned Riemannian Metric', fontsize=12)
ax3.legend()
ax3.set_aspect('equal')

# Summary
ax4 = axes[1, 1]
ax4.axis('off')
summary = "Summary (last 50 episodes)\n" + "="*35 + "\n\n"
for method in ['PPO', 'SGPO']:
    ret = np.mean(results[method]['returns'][-50:])
    viol = np.mean(results[method]['violations'][-50:])
    total = sum(results[method]['violations'])
    summary += f"{method}:\n  Return: {ret:.1f}\n  Violations: {viol:.1f}\n  Total: {total}\n\n"

improvement = (sum(results['PPO']['violations']) - sum(results['SGPO']['violations'])) / max(sum(results['PPO']['violations']), 1) * 100
summary += f"\nSGPO: {improvement:.0f}% fewer violations"
ax4.text(0.1, 0.9, summary, transform=ax4.transAxes, fontsize=12, verticalalignment='top', fontfamily='monospace')

plt.tight_layout()
plt.savefig('gpo_results.png', dpi=150)
plt.show()

## Modal.com Deployment (Optional)

If you want to run this on Modal with a GPU:

In [ ]:
# Uncomment to run on Modal
"""
import modal

app = modal.App("gpo-training")

image = modal.Image.debian_slim().pip_install(
    "torch", "numpy", "matplotlib"
)

@app.function(gpu="T4", image=image, timeout=3600)
def train_on_gpu(n_episodes: int = 500):
    import torch
    # ... paste the training code here ...
    results, metric, hazards = run_experiment(n_episodes=n_episodes)
    return results

@app.local_entrypoint()
def main():
    results = train_on_gpu.remote(n_episodes=500)
    print(f"PPO violations: {sum(results['PPO']['violations'])}")
    print(f"SGPO violations: {sum(results['SGPO']['violations'])}")
"""

## Upstream PR Notes

The PR proposals for gymnasium/safety-gymnasium are documented in:
`src/environments/UPSTREAM_CONTRIBUTIONS.md`

Key contributions:
1. **Safety-Gymnasium**: `RiemannianWrapper` with geodesic direction hints
2. **LMRL-Gym**: `SafetyConstraint` system for conversational RL
3. **Robust-Gymnasium**: `RiemannianAdversary` that exploits low-metric states
4. **TextWorld/TALES**: `IrreversibilityDetector` + `WinnabilityTracker`